In [1]:
filename = "stack4.mp4"

In [2]:
import cv2
import mediapipe as mp
from mediapipe.tasks.python import vision
from mediapipe.tasks.python.vision import drawing_utils, drawing_styles
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.animation as animation
import urllib.request
import os
import json
from contextlib import ExitStack
from IPython.display import HTML, display

# --- Model paths and download ---
HAND_MODEL_PATH = 'hand_landmarker.task'
POSE_MODEL_PATH = 'pose_landmarker.task'
FACE_MODEL_PATH = 'face_landmarker.task'

if not os.path.exists(HAND_MODEL_PATH):
    urllib.request.urlretrieve(
        'https://storage.googleapis.com/mediapipe-models/hand_landmarker/hand_landmarker/float16/1/hand_landmarker.task',
        HAND_MODEL_PATH
    )
if not os.path.exists(POSE_MODEL_PATH):
    urllib.request.urlretrieve(
        'https://storage.googleapis.com/mediapipe-models/pose_landmarker/pose_landmarker_heavy/float16/1/pose_landmarker_heavy.task',
        POSE_MODEL_PATH
    )
if not os.path.exists(FACE_MODEL_PATH):
    urllib.request.urlretrieve(
        'https://storage.googleapis.com/mediapipe-models/face_landmarker/face_landmarker/float16/1/face_landmarker.task',
        FACE_MODEL_PATH
    )

# --- Task options (VIDEO mode for per-frame processing) ---
BaseOptions = mp.tasks.BaseOptions
HandLandmarker = mp.tasks.vision.HandLandmarker
HandLandmarkerOptions = mp.tasks.vision.HandLandmarkerOptions
PoseLandmarker = mp.tasks.vision.PoseLandmarker
PoseLandmarkerOptions = mp.tasks.vision.PoseLandmarkerOptions
FaceLandmarker = mp.tasks.vision.FaceLandmarker
FaceLandmarkerOptions = mp.tasks.vision.FaceLandmarkerOptions
VisionRunningMode = mp.tasks.vision.RunningMode

def create_base_options(model_path, delegate):
    return BaseOptions(model_asset_path=model_path, delegate=delegate)


def build_hand_options(delegate):
    return HandLandmarkerOptions(
        base_options=create_base_options(HAND_MODEL_PATH, delegate),
        running_mode=VisionRunningMode.VIDEO,
        num_hands=2,
        min_hand_detection_confidence=0.5,
        min_hand_presence_confidence=0.5,
        min_tracking_confidence=0.5,
    )


def build_pose_options(delegate):
    return PoseLandmarkerOptions(
        base_options=create_base_options(POSE_MODEL_PATH, delegate),
        running_mode=VisionRunningMode.VIDEO,
        min_pose_detection_confidence=0.5,
        min_pose_presence_confidence=0.5,
        min_tracking_confidence=0.5,
    )


def build_face_options(delegate):
    return FaceLandmarkerOptions(
        base_options=create_base_options(FACE_MODEL_PATH, delegate),
        running_mode=VisionRunningMode.VIDEO,
        num_faces=1,
        min_face_detection_confidence=0.5,
        min_face_presence_confidence=0.5,
        min_tracking_confidence=0.5,
    )


def create_landmarker(factory, options_builder, name):
    delegates = [BaseOptions.Delegate.GPU, BaseOptions.Delegate.CPU]
    errors = []
    for delegate in delegates:
        try:
            return factory.create_from_options(options_builder(delegate)), delegate
        except (OSError, RuntimeError) as exc:
            errors.append(f'{delegate.name}: {exc}')

    raise RuntimeError(f"Could not create {name}. Attempts: {' | '.join(errors)}")


def process_frame_hand(frame_bgr, hand_landmarker, frame_timestamp_ms):
    frame_rgb = cv2.cvtColor(frame_bgr, cv2.COLOR_BGR2RGB)
    mp_image = mp.Image(image_format=mp.ImageFormat.SRGB, data=frame_rgb)
    return hand_landmarker.detect_for_video(mp_image, frame_timestamp_ms)


def process_frame_pose(frame_bgr, pose_landmarker, frame_timestamp_ms):
    frame_rgb = cv2.cvtColor(frame_bgr, cv2.COLOR_BGR2RGB)
    mp_image = mp.Image(image_format=mp.ImageFormat.SRGB, data=frame_rgb)
    return pose_landmarker.detect_for_video(mp_image, frame_timestamp_ms)


def process_frame_face(frame_bgr, face_landmarker, frame_timestamp_ms):
    frame_rgb = cv2.cvtColor(frame_bgr, cv2.COLOR_BGR2RGB)
    mp_image = mp.Image(image_format=mp.ImageFormat.SRGB, data=frame_rgb)
    return face_landmarker.detect_for_video(mp_image, frame_timestamp_ms)



def serialize_landmark(landmark):
    landmark_data = {
        'x': float(landmark.x),
        'y': float(landmark.y),
        'z': float(landmark.z),
    }
    if hasattr(landmark, 'visibility'):
        landmark_data['visibility'] = float(landmark.visibility) if landmark.visibility else 0
    if hasattr(landmark, 'presence'):
        landmark_data['presence'] = float(landmark.presence) if landmark.visibility else 0
    return landmark_data

def serialize_landmark_list(landmark_list):
    return [serialize_landmark(landmark) for landmark in landmark_list]

def serialize_categories(category_list):
    return [
        {
            'index': category.index,
            'score': float(category.score),
            'category_name': category.category_name,
            'display_name': category.display_name,
        }
        for category in category_list
    ]

def serialize_hand_result(hand_result):
    hand_landmarks = getattr(hand_result, "hand_landmarks", []) or []
    handedness = getattr(hand_result, "handedness", []) or []
    return [
        {
            'landmarks': serialize_landmark_list(landmarks),
            'handedness': serialize_categories(handedness[i]) if i < len(handedness) else [],
        }
        for i, landmarks in enumerate(hand_landmarks)
    ]

def serialize_pose_result(pose_result):
    pose_landmarks = getattr(pose_result, "pose_landmarks", []) or []
    return [serialize_landmark_list(landmarks) for landmarks in pose_landmarks]

def serialize_face_result(face_result):
    face_landmarks = getattr(face_result, "face_landmarks", []) or []
    return [serialize_landmark_list(landmarks) for landmarks in face_landmarks]

def draw_hand(bgr_image, hand_result, in_place=False):
    if in_place:
        annotated = bgr_image
    else:
        annotated = np.copy(bgr_image)

    if hasattr(hand_result, 'hand_landmarks') and hand_result.hand_landmarks:
        hand_style = drawing_styles.get_default_hand_landmarks_style()
        hand_conn_style = drawing_styles.get_default_hand_connections_style()
        for landmarks in hand_result.hand_landmarks:
            drawing_utils.draw_landmarks(
                image=annotated,
                landmark_list=landmarks,
                connections=vision.HandLandmarksConnections.HAND_CONNECTIONS,
                landmark_drawing_spec=hand_style,
                connection_drawing_spec=hand_conn_style
            )
    return annotated

def draw_pose(bgr_image, pose_result, in_place=False):
    if in_place:
        annotated = bgr_image
    else:
        annotated = np.copy(bgr_image)

    if hasattr(pose_result, 'pose_landmarks') and pose_result.pose_landmarks:
        pose_style = drawing_styles.get_default_pose_landmarks_style()
        pose_conn_style = drawing_utils.DrawingSpec(color=(0, 255, 0), thickness=2)
        for landmarks in pose_result.pose_landmarks:
            drawing_utils.draw_landmarks(
                image=annotated,
                landmark_list=landmarks,
                connections=vision.PoseLandmarksConnections.POSE_LANDMARKS,
                landmark_drawing_spec=pose_style,
                connection_drawing_spec=pose_conn_style
            )
    return annotated

def draw_face(bgr_image, face_result, in_place=False):
    if in_place:
        annotated = bgr_image
    else:
        annotated = np.copy(bgr_image)

    if hasattr(face_result, 'face_landmarks') and face_result.face_landmarks:
        tesselation_style = drawing_styles.get_default_face_mesh_tesselation_style()
        contours_style = drawing_styles.get_default_face_mesh_contours_style()
        iris_style = drawing_styles.get_default_face_mesh_iris_connections_style()
        for landmarks in face_result.face_landmarks:
            drawing_utils.draw_landmarks(
                image=annotated,
                landmark_list=landmarks,
                connections=vision.FaceLandmarksConnections.FACE_LANDMARKS_TESSELATION,
                landmark_drawing_spec=None,
                connection_drawing_spec=tesselation_style
            )
            drawing_utils.draw_landmarks(
                image=annotated,
                landmark_list=landmarks,
                connections=vision.FaceLandmarksConnections.FACE_LANDMARKS_CONTOURS,
                landmark_drawing_spec=None,
                connection_drawing_spec=contours_style
            )
            drawing_utils.draw_landmarks(
                image=annotated,
                landmark_list=landmarks,
                connections=vision.FaceLandmarksConnections.FACE_LANDMARKS_LEFT_IRIS,
                landmark_drawing_spec=None,
                connection_drawing_spec=iris_style
            )
            drawing_utils.draw_landmarks(
                image=annotated,
                landmark_list=landmarks,
                connections=vision.FaceLandmarksConnections.FACE_LANDMARKS_RIGHT_IRIS,
                landmark_drawing_spec=None,
                connection_drawing_spec=iris_style
            )
    return annotated


# --- Demo: load video and process with hand + pose + face ---
file_path = filename
cap = cv2.VideoCapture(file_path)
if not cap.isOpened():
    raise FileNotFoundError(f'Could not open video file: {file_path}')
fps = cap.get(cv2.CAP_PROP_FPS) or 30.0

original_frames = []
hand_frames = []
hand_pose_frames = []
hand_face_frames = []
all_frames = []
keypoints_output_path = f"{filename.split('.')[0]}.json"
keypoints_by_frame = []

with ExitStack() as stack:
    hand_landmarker, hand_delegate = create_landmarker(HandLandmarker, build_hand_options, 'hand landmarker')
    pose_landmarker, pose_delegate = create_landmarker(PoseLandmarker, build_pose_options, 'pose landmarker')
    face_landmarker, face_delegate = create_landmarker(FaceLandmarker, build_face_options, 'face landmarker')

    hand_landmarker = stack.enter_context(hand_landmarker)
    pose_landmarker = stack.enter_context(pose_landmarker)
    face_landmarker = stack.enter_context(face_landmarker)

    print(
        f'Delegates used -> hand: {hand_delegate.name}, pose: {pose_delegate.name}, face: {face_delegate.name}'
    )

    frame_index = 0
    while cap.isOpened():
        ret, frame = cap.read()
        if not ret:
            break
        ts_ms = int((frame_index / fps) * 1000)
        frame_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        hand_result = process_frame_hand(frame, hand_landmarker, ts_ms)
        pose_result = process_frame_pose(frame, pose_landmarker, ts_ms)
        face_result = process_frame_face(frame, face_landmarker, ts_ms)

        # 1. Original
        original = frame_rgb

        # 2. Hand only
        img_hand = draw_hand(frame, hand_result)
        img_hand = cv2.cvtColor(img_hand, cv2.COLOR_BGR2RGB)

        # 3. Hand + Body
        img_hand_pose = draw_hand(frame, hand_result)
        img_hand_pose = draw_pose(img_hand_pose, pose_result, in_place=True)
        img_hand_pose = cv2.cvtColor(img_hand_pose, cv2.COLOR_BGR2RGB)

        # 4. Hand + Face
        img_hand_face = draw_hand(frame, hand_result)
        img_hand_face = draw_face(img_hand_face, face_result, in_place=True)
        img_hand_face = cv2.cvtColor(img_hand_face, cv2.COLOR_BGR2RGB)

        # 5. Hand + Body + Face
        img_all = draw_hand(frame, hand_result)
        img_all = draw_pose(img_all, pose_result, in_place=True)
        img_all = draw_face(img_all, face_result, in_place=True)
        img_all = cv2.cvtColor(img_all, cv2.COLOR_BGR2RGB)

        keypoints_by_frame.append({
            'frame_index': frame_index,
            'timestamp_ms': ts_ms,
            'hand_landmarks': serialize_hand_result(hand_result),
            'pose_landmarks': serialize_pose_result(pose_result),
            'face_landmarks': serialize_face_result(face_result),
        })

        original_frames.append(original)
        hand_frames.append(img_hand)
        hand_pose_frames.append(img_hand_pose)
        hand_face_frames.append(img_hand_face)
        all_frames.append(img_all)

        frame_index += 1

cap.release()

with open(keypoints_output_path, "w", encoding="utf-8") as f:
    json.dump({
        'video_path': file_path,
        'fps': float(fps),
        'num_frames': frame_index,
        'frames': keypoints_by_frame,
    }, f, indent=2)

print(f'Saved keypoints to {keypoints_output_path}')

original_frames = np.array(original_frames)
hand_frames = np.array(hand_frames)
hand_pose_frames = np.array(hand_pose_frames)
hand_face_frames = np.array(hand_face_frames)
all_frames = np.array(all_frames)
num_frames = len(original_frames)
if num_frames == 0:
    raise ValueError(f'No frames were read from {file_path}')

# Prepare the figure with 1 row, 5 columns
fig, axs = plt.subplots(1, 5, figsize=(20, 4))
titles = ['Original', 'Hand', 'Hand + Body', 'Hand + Face', 'Hand + Body + Face']

# The images in the axes
ims = [axs[0].imshow(original_frames[0]),
       axs[1].imshow(hand_frames[0]),
       axs[2].imshow(hand_pose_frames[0]),
       axs[3].imshow(hand_face_frames[0]),
       axs[4].imshow(all_frames[0])]

for ax, title in zip(axs, titles):
    ax.set_title(title)
    ax.axis('off')

def update(frame_index):
    ims[0].set_array(original_frames[frame_index])
    ims[1].set_array(hand_frames[frame_index])
    ims[2].set_array(hand_pose_frames[frame_index])
    ims[3].set_array(hand_face_frames[frame_index])
    ims[4].set_array(all_frames[frame_index])
    return ims

ani = animation.FuncAnimation(
    fig,
    update,
    frames=num_frames,
    interval=50,   # Delay between frames in milliseconds (20 fps)
    blit=True,
    repeat=True
)

plt.tight_layout()
plt.close(fig)
display(HTML(ani.to_jshtml()))


libEGL warning: DRI3 error: Could not get DRI3 device
libEGL warning: Ensure your X server supports DRI3 to get accelerated rendering
I0000 00:00:1776925166.204652     989 gl_context_egl.cc:85] Successfully initialized EGL. Major : 1 Minor: 5
I0000 00:00:1776925166.301022    1028 gl_context.cc:385] GL version: 3.2 (OpenGL ES 3.2 Mesa 25.2.8-0ubuntu0.24.04.1), renderer: llvmpipe (LLVM 20.1.2, 256 bits)
I0000 00:00:1776925167.816596    1035 gl_context_egl.cc:85] Successfully initialized EGL. Major : 1 Minor: 5
I0000 00:00:1776925167.943432    1048 gl_context.cc:385] GL version: 3.2 (OpenGL ES 3.2 Mesa 25.2.8-0ubuntu0.24.04.1), renderer: llvmpipe (LLVM 20.1.2, 256 bits)
W0000 00:00:1776925173.313628    1064 face_landmarker_graph.cc:174] Sets FaceBlendshapesGraph acceleration to xnnpack by default.
I0000 00:00:1776925173.349598    1064 gl_context_egl.cc:85] Successfully initialized EGL. Major : 1 Minor: 5
I0000 00:00:1776925173.370829    1077 gl_context.cc:385] GL version: 3.2 (OpenGL ES 3

Delegates used -> hand: GPU, pose: GPU, face: GPU


W0000 00:00:1776925175.138861     990 landmark_projection_calculator.cc:78] Using NORM_RECT without IMAGE_DIMENSIONS is only supported for the square ROI. Provide IMAGE_DIMENSIONS or use PROJECTION_MATRIX.
E0000 00:00:1776925177.177704    1028 tensor.cc:404] Tensors are designed for single writes. Multiple writes to a Tensor instance are not supported and may lead to undefined behavior due to lack of synchronization.


Saved keypoints to stack4.json


Animation size has reached 21180393 bytes, exceeding the limit of 20971520.0. If you're sure you want a larger animation embedded, set the animation.embed_limit rc parameter to a larger value (in MB). This and further frames will be dropped.


In [3]:
# 3D HTML animation from keypoints.json with six camera views
import json
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.animation as animation
from IPython.display import HTML, display

HAND_CONNECTIONS = [
    (0, 1), (1, 2), (2, 3), (3, 4),
    (0, 5), (5, 6), (6, 7), (7, 8),
    (5, 9), (9, 10), (10, 11), (11, 12),
    (9, 13), (13, 14), (14, 15), (15, 16),
    (13, 17), (17, 18), (18, 19), (19, 20),
    (0, 17)
]

POSE_CONNECTIONS = [
    (0, 1), (1, 2), (2, 3), (3, 7),
    (0, 4), (4, 5), (5, 6), (6, 8),
    (9, 10),
    (11, 12),
    (11, 13), (13, 15),
    (12, 14), (14, 16),
    (15, 17), (15, 19), (15, 21),
    (16, 18), (16, 20), (16, 22),
    (11, 23), (12, 24),
    (23, 24),
    (23, 25), (25, 27), (27, 29), (29, 31),
    (24, 26), (26, 28), (28, 30), (30, 32)
]

EXPECTED_HAND_POINTS = 21
EXPECTED_POSE_POINTS = 33
EXPECTED_FACE_POINTS = 478
MISSING_POINT = {
    'x': np.nan,
    'y': np.nan,
    'z': np.nan,
    'visibility': 0.0,
    'presence': 0.0,
}

VIEW_SPECS = [
    ('X+', 0, 0),
    ('X-', 0, 180),
    ('Y+', 0, 90),
    ('Y-', 0, -90),
    ('Z+', 90, -90),
    ('Z-', -90, -90),
]

with open(f"{filename.split('.')[0]}.json", 'r', encoding='utf-8') as f:
    keypoints_data = json.load(f)

frames_data = keypoints_data.get('frames', [])
if not frames_data:
    raise ValueError('No frames found in keypoints.json')

def normalize_landmarks(landmarks, expected_len, fallback=None):
    points = list(landmarks or [])
    if not points and fallback is not None:
        points = [dict(point) for point in fallback]
    if len(points) < expected_len:
        points = points + [dict(MISSING_POINT) for _ in range(expected_len - len(points))]
    elif len(points) > expected_len:
        points = points[:expected_len]
    normalized = []
    for point in points:
        normalized.append({
            'x': float(point.get('x', np.nan)),
            'y': float(point.get('y', np.nan)),
            'z': float(point.get('z', np.nan)),
            'visibility': float(point.get('visibility', 0.0)),
            'presence': float(point.get('presence', 0.0)),
        })
    return normalized

def extract_xyz(landmarks):
    xs = np.array([float(point['x']) for point in landmarks], dtype=float)
    ys = -np.array([float(point['y']) for point in landmarks], dtype=float)
    zs = -np.array([float(point['z']) for point in landmarks], dtype=float)
    return xs, ys, zs

def build_connection_coords(landmarks, connections):
    x_lines, y_lines, z_lines = [], [], []
    for start_idx, end_idx in connections:
        if start_idx >= len(landmarks) or end_idx >= len(landmarks):
            continue
        start = landmarks[start_idx]
        end = landmarks[end_idx]
        if any(np.isnan([start['x'], start['y'], start['z'], end['x'], end['y'], end['z']])):
            x_lines.extend([np.nan, np.nan, np.nan])
            y_lines.extend([np.nan, np.nan, np.nan])
            z_lines.extend([np.nan, np.nan, np.nan])
            continue
        x_lines.extend([float(start['x']), float(end['x']), np.nan])
        y_lines.extend([-float(start['y']), -float(end['y']), np.nan])
        z_lines.extend([-float(start['z']), -float(end['z']), np.nan])
    return x_lines, y_lines, z_lines

def classify_hands(hand_entries):
    hands = {'left': [], 'right': []}
    for entry in hand_entries:
        handedness = entry.get('handedness', [])
        label = 'unknown'
        if handedness:
            category_name = (handedness[0].get('category_name') or '').lower()
            if category_name in hands:
                label = category_name
        landmarks = entry.get('landmarks', [])
        if label in hands and not hands[label]:
            hands[label] = landmarks
        elif not hands['left']:
            hands['left'] = landmarks
        elif not hands['right']:
            hands['right'] = landmarks
    return hands

def preprocess_frames(frames_data):
    processed = []
    last_valid = {
        'left_hand': None,
        'right_hand': None,
        'pose': None,
        'face': None,
    }
    for frame_data in frames_data:
        hands = classify_hands(frame_data.get('hand_landmarks', []))
        pose_landmarks = frame_data.get('pose_landmarks', [])
        face_landmarks = frame_data.get('face_landmarks', [])

        left_hand = normalize_landmarks(hands.get('left', []), EXPECTED_HAND_POINTS, fallback=last_valid['left_hand'])
        right_hand = normalize_landmarks(hands.get('right', []), EXPECTED_HAND_POINTS, fallback=last_valid['right_hand'])
        pose = normalize_landmarks(pose_landmarks[0] if pose_landmarks else [], EXPECTED_POSE_POINTS, fallback=last_valid['pose'])
        face = normalize_landmarks(face_landmarks[0] if face_landmarks else [], EXPECTED_FACE_POINTS, fallback=last_valid['face'])

        if not np.isnan([point['x'] for point in left_hand]).all():
            last_valid['left_hand'] = [dict(point) for point in left_hand]
        if not np.isnan([point['x'] for point in right_hand]).all():
            last_valid['right_hand'] = [dict(point) for point in right_hand]
        if not np.isnan([point['x'] for point in pose]).all():
            last_valid['pose'] = [dict(point) for point in pose]
        if not np.isnan([point['x'] for point in face]).all():
            last_valid['face'] = [dict(point) for point in face]

        processed.append({
            'frame_index': frame_data.get('frame_index', len(processed)),
            'hands': {'left': left_hand, 'right': right_hand},
            'pose': pose,
            'face': face,
        })
    return processed

def gather_bounds(processed_frames):
    xs_all, ys_all, zs_all = [], [], []
    for frame_data in processed_frames:
        for landmarks in [frame_data['hands']['left'], frame_data['hands']['right'], frame_data['pose'], frame_data['face']]:
            xs, ys, zs = extract_xyz(landmarks)
            valid = ~(np.isnan(xs) | np.isnan(ys) | np.isnan(zs))
            if np.any(valid):
                xs_all.extend(xs[valid].tolist())
                ys_all.extend(ys[valid].tolist())
                zs_all.extend(zs[valid].tolist())
    if not xs_all:
        return (-1.0, 1.0), (-1.0, 1.0), (-1.0, 1.0)
    x_min, x_max = min(xs_all), max(xs_all)
    y_min, y_max = min(ys_all), max(ys_all)
    z_min, z_max = min(zs_all), max(zs_all)
    span = max(x_max - x_min, y_max - y_min, z_max - z_min, 1e-6)
    x_mid = (x_min + x_max) / 2
    y_mid = (y_min + y_max) / 2
    z_mid = (z_min + z_max) / 2
    half = span / 2
    pad = span * 0.05
    return (x_mid - half - pad, x_mid + half + pad), (y_mid - half - pad, y_mid + half + pad), (z_mid - half - pad, z_mid + half + pad)

def style_axis(ax, title, bounds, elev, azim):
    x_limits, y_limits, z_limits = bounds
    ax.set_xlim(*x_limits)
    ax.set_ylim(*y_limits)
    ax.set_zlim(*z_limits)
    ax.set_xlabel('X')
    ax.set_ylabel('Y')
    ax.set_zlabel('Z')
    ax.set_title(title)
    ax.view_init(elev=elev, azim=azim)
    try:
        ax.set_box_aspect((1, 1, 1))
    except AttributeError:
        pass

def draw_landmark_group(ax, landmarks, color, size, connections=None, line_width=1.5):
    xs, ys, zs = extract_xyz(landmarks)
    valid = ~(np.isnan(xs) | np.isnan(ys) | np.isnan(zs))
    if np.any(valid):
        ax.scatter(xs[valid], ys[valid], zs[valid], c=color, s=size, depthshade=False)
    if connections:
        x_lines, y_lines, z_lines = build_connection_coords(landmarks, connections)
        ax.plot(x_lines, y_lines, z_lines, color=color, linewidth=line_width, alpha=0.9)

def draw_frame(ax, frame_data):
    draw_landmark_group(ax, frame_data['hands']['left'], '#d62728', 12, HAND_CONNECTIONS, 1.8)
    draw_landmark_group(ax, frame_data['hands']['right'], '#2ca02c', 12, HAND_CONNECTIONS, 1.8)
    draw_landmark_group(ax, frame_data['pose'], '#1f77b4', 10, POSE_CONNECTIONS, 1.8)
    draw_landmark_group(ax, frame_data['face'], '#9467bd', 3)

processed_frames = preprocess_frames(frames_data)
bounds = gather_bounds(processed_frames)
fig, axes = plt.subplots(2, 3, figsize=(15, 10), subplot_kw={'projection': '3d'})
axes = axes.ravel()

def update_3d(frame_index):
    frame_data = processed_frames[frame_index]
    for ax, (title, elev, azim) in zip(axes, VIEW_SPECS):
        ax.cla()
        style_axis(ax, title, bounds, elev, azim)
        draw_frame(ax, frame_data)
    fig.suptitle(f"Hand + Body + Face | Frame {frame_data['frame_index']}")
    return []

update_3d(0)
ani_3d = animation.FuncAnimation(
    fig,
    update_3d,
    frames=len(processed_frames),
    interval=50,
    blit=False,
    repeat=True,
)

plt.tight_layout()
plt.close(fig)
display(HTML(ani_3d.to_jshtml()))
from matplotlib.animation import PillowWriter
ani_3d.save(f"{filename.split('.')[0]}.gif", writer=PillowWriter(fps=30))

